# VPR Project Shared Notebook

This notebook is used to:
- access the shared Drive folder in order to read datasets and save outputs
- clone the github repository of the project to get the necessary python code
- install dependencies necessary to run the code
- run VPR evaluation
- run image matching
- run reranking

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === Base paths ===
DRIVE_ROOT   = "/content/drive/MyDrive/VPR"

DATASET_DIR    = f"{DRIVE_ROOT}/datasets"
LOG_DIR        = f"{DRIVE_ROOT}/logs"
ADAPTIVE_ROOT  = f"{DRIVE_ROOT}/adaptive"
REGRESSORS_DIR = f"{DRIVE_ROOT}/regressors"

for d in [DATASET_DIR, LOG_DIR, ADAPTIVE_ROOT, REGRESSORS_DIR]:
    os.makedirs(d, exist_ok=True)

print("DATASET_DIR    =", DATASET_DIR)
print("LOG_DIR        =", LOG_DIR)
print("ADAPTIVE_ROOT  =", ADAPTIVE_ROOT)
print("REGRESSORS_DIR =", REGRESSORS_DIR)
print()
print("Carica i file JSON dei regressori in:", REGRESSORS_DIR)

In [ ]:
# clone github repository (code source):
!git clone --recursive https://github.com/roccnroll/Visual-Place-Recognition-Project.git

# change directory to the new folder:
%cd Visual-Place-Recognition-Project

In [ ]:
# controllo per verificare se è necessario scaricare nuovamente le dependencies (cella sottostante)
try:
    import faiss
    import matching
    print("Dependencies already installed.")
except Exception as e:
    print("Missing dependency:", e)

Missing dependency: No module named 'faiss'


In [ ]:
# install dependencies:
%cd /content/Visual-Place-Recognition-Project/image-matching-models
!pip install -e .
!pip install faiss-cpu

/content/Visual-Place-Recognition-Project/image-matching-models
Obtaining file:///content/Visual-Place-Recognition-Project/image-matching-models
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.3/225.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# after the installation of dependencies go back to original directory:
%cd /content/Visual-Place-Recognition-Project

/content/Visual-Place-Recognition-Project


## Dataset download
!! Run this only when needed !!

In [ ]:
!python download_datasets.py

Downloading...
From (original): https://drive.google.com/uc?id=15QB3VNKj93027UAQWv7pzFQO1JDCdZj2
From (redirected): https://drive.google.com/uc?id=15QB3VNKj93027UAQWv7pzFQO1JDCdZj2&confirm=t&uuid=caf994e5-6382-44bb-a5ca-6ad8a7699797
To: /content/Visual-Place-Recognition-Project/data/tokyo_xs.zip
100% 141M/141M [00:01<00:00, 134MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1tQqEyt3go3vMh4fj_LZrRcahoTbzzH-y
From (redirected): https://drive.google.com/uc?id=1tQqEyt3go3vMh4fj_LZrRcahoTbzzH-y&confirm=t&uuid=cf4f5341-12d4-4798-98a8-7dc7b255bede
To: /content/Visual-Place-Recognition-Project/data/sf_xs.zip
100% 1.03G/1.03G [00:15<00:00, 67.6MB/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 273, in download
    url = get_url_from_gdrive_confirmation(res.text)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 55, in get_url_from_gdrive

## First VPR evaluation

Repo's readme used the following intialization of parameters:

device = cpu --> serve per forzare l'uso della cpu locale (da modificare per run più grandi)

num_workers = 8 -->  numero di processi paralleli vengono usati dal dataloader per caricare/preprocessare le immagini.


batch_size = 32 --> quante immagini il modello processa insieme in ogni forward pass.


log_dir --> specifica dove salvare i log e gli output della run.


method = cosplace --backbone=ResNet18 descriptors_dimension=512 --> questi parametri definiscono il modello VPR da usare.


image_size 512 = 512 --> imposta la dimensione a cui le immagini vengono ridimensionate.


database_folder '' --> da compilare per indicare esplicitamente dove stanno le immagini database e query.

!!! PER ORA USATO TOKYO MA SENZA UNA RATIO, VEDERE COSA DICE NEL PDF SUI DATABASE !!!\


queries_folder '' --> da compilare per indicare esplicitamente dove stanno le immagini database e query.


num_preds_to_save = 20 --> quante predizioni per query salvare (per ora ho messo meno perché il matching locale è il pezzo più costoso).


recall_values 1 5 10 20 --> specifica a quali valori di k calcolare la recall.


save_for_uncertainty --> potrebbe essere usato per salvare output aggiuntivi utili per una fase di uncertainty evaluation.


code in the following cell might have different values to ease the computation for local CPU or some fields might be ommissed with the only intention of facilitating the first trial run.

In [ ]:
!python VPR-methods-evaluation/main.py \
    --device cuda \
    --num_workers 4 \
    --batch_size 32 \
    --num_preds_to_save 20 \
    --log_dir {LOG_DIR} \
    --method cosplace \
    --backbone ResNet18 \
    --descriptors_dimension 512 \
    --image_size 512 512 \
    --database_folder /content/Visual-Place-Recognition-Project/data/sf_xs/val/database \
    --queries_folder /content/Visual-Place-Recognition-Project/data/sf_xs/val/queries \
    --recall_values 1 5 10 20 \
    --save_for_uncertainty

# Adaptive Re-Ranking Pipeline

La pipeline adattiva decide **per ogni query** se rerankare o meno, usando un regressore logistico calibrato offline.

**Step della pipeline:**
1. **Estrazione SU** — calcola RS, SD, SU da `z_data.torch` → `su_scores.csv`
2. **Estrazione inliers** *(solo se feature-set include inliers)* — IM top-1 su tutte le query → `inliers.csv`
3. **Selezione query** — applica criterio di probabilità → copia `.txt` selezionati in `preds_filtered/`
4. **Reranking completo** — IM top-20 solo sulle query selezionate
5. **Check performance** — R@1 finale combinando query rerankate e skippate

**Parametri da scegliere:**
- `FEATURE_SET`: `inliers` | `RS` | `SD` | `SU` | `SU+inliers`
- `CRITERION`: `P(hard)` | `P(help)` | `P(help)-aP(hurts)` | `P(help)/P(hurts)>1`
- `MATCHER`: `sp-lg` | `loftr`

In [ ]:
import os

# ============================================================
# CONFIGURAZIONE — cambia questi valori ad ogni run
# ============================================================

# Cartella creata da main.py (usa il timestamp che trovi in Drive/VPR/logs/)
RUN_DIR      = "/content/drive/MyDrive/VPR/logs/2026-07-01_01-49-23"

# Etichetta per la cartella adattiva (puoi mettere quello che vuoi)
RUN_NAME     = "Cosplace_sf_xs"

FEATURE_SET  = "SU"                 # inliers | RS | SD | SU | SU+inliers
CRITERION    = "P(help)"            # P(hard) | P(help) | P(help)-aP(hurts) | P(help)/P(hurts)>1
MATCHER      = "superpoint-lg"      # superpoint-lg | loftr
MODEL        = "cosplace"           # cosplace | megaloc

# Regressori dalla repo
REPO_ROOT       = "/content/Visual-Place-Recognition-Project"
REGRESSORS_JSON = f"{REPO_ROOT}/regressors/{MODEL}_regressors.json"

# ============================================================
# Percorsi derivati automaticamente
# ============================================================

DRIVE_ROOT   = "/content/drive/MyDrive/VPR"
PREDS_DIR    = f"{RUN_DIR}/preds"
Z_DATA_PATH  = f"{RUN_DIR}/z_data.torch"

ADAPTIVE_DIR   = f"{DRIVE_ROOT}/adaptive/{RUN_NAME}_{FEATURE_SET}_{CRITERION.replace('(','').replace(')','').replace('/','_')}"
FEATURES_DIR   = f"{ADAPTIVE_DIR}/features"
PREDS_FILTERED = f"{ADAPTIVE_DIR}/preds_filtered"
RERANKED_DIR   = f"{ADAPTIVE_DIR}/reranked"

for d in [FEATURES_DIR, PREDS_FILTERED, RERANKED_DIR]:
    os.makedirs(d, exist_ok=True)

print("RUN_DIR         =", RUN_DIR)
print("PREDS_DIR       =", PREDS_DIR)
print("Z_DATA_PATH     =", Z_DATA_PATH)
print("FEATURE_SET     =", FEATURE_SET)
print("CRITERION       =", CRITERION)
print("REGRESSORS_JSON =", REGRESSORS_JSON)
print("ADAPTIVE_DIR    =", ADAPTIVE_DIR)

## Run Pipeline
Esegui questa cella per lanciare automaticamente tutti gli step.
La pipeline decide da sola quali estrazioni fare in base a `FEATURE_SET`.

In [ ]:
import subprocess, shlex

REPO = "/content/Visual-Place-Recognition-Project"

def run_step(label, cmd):
    print(f"\n{'='*60}")
    print(f">> {label}")
    print('='*60)
    result = subprocess.run(shlex.split(cmd))
    if result.returncode != 0:
        raise RuntimeError(f"Step fallito: {label}")

# Step 1: SU (solo se FEATURE_SET usa RS/SD/SU)
if FEATURE_SET in ("RS", "SD", "SU", "SU+inliers"):
    run_step("Step 1 - Estrazione SU",
        f"python {REPO}/VPR-Adaptive-ReRanking/extract_su.py"
        f" --z-data {Z_DATA_PATH}"
        f" --output-dir {FEATURES_DIR}")
else:
    print("Step 1 saltato")

# Step 2: inliers (solo se FEATURE_SET usa inliers)
if FEATURE_SET in ("inliers", "SU+inliers"):
    run_step("Step 2 - Estrazione inliers (IM top-1)",
        f"python {REPO}/VPR-Adaptive-ReRanking/extract_inliers.py"
        f" --preds-dir {PREDS_DIR}"
        f" --output-dir {FEATURES_DIR}"
        f" --matcher {MATCHER}"
        f" --device cuda")
else:
    print("Step 2 saltato")

# Step 3: selezione query
run_step("Step 3 - Selezione query",
    f"python {REPO}/VPR-Adaptive-ReRanking/select_queries.py"
    f" --features-dir {FEATURES_DIR}"
    f" --preds-dir {PREDS_DIR}"
    f" --regressors-json {REGRESSORS_JSON}"
    f" --matcher {MATCHER}"
    f" --feature-set {FEATURE_SET}"
    f" --criterion {CRITERION}"
    f" --output-dir {PREDS_FILTERED}"
    f" --skipped-file {ADAPTIVE_DIR}/skipped.txt")

# Step 4: reranking completo
run_step("Step 4 - Reranking (IM top-20)",
    f"python {REPO}/match_queries_preds.py"
    f" --preds-dir {PREDS_FILTERED}"
    f" --out-dir {RERANKED_DIR}"
    f" --matcher {MATCHER}"
    f" --device cuda"
    f" --im-size 512"
    f" --num-preds 20")

# Step 5: performance finale
run_step("Step 5 - Check performance",
    f"python {REPO}/VPR-Adaptive-ReRanking/check_performance.py"
    f" --preds-dir {PREDS_DIR}"
    f" --reranked-dir {RERANKED_DIR}"
    f" --skipped-file {ADAPTIVE_DIR}/skipped.txt"
    f" --recall-values 1 5 10 20")

## Sequential Pipeline
Pipeline a tre cancelli in cascata (gate1 → gate5 → gate10):
1. **Estrazione feature** — IM top-10 su TUTTE le query → 
2. **Selezione sequenziale** — applica i tre gate in cascata → solo le query che superano tutti e tre vanno a IM top-20
3. **Reranking** — IM top-20 sulle query selezionate
4. **Performance** — R@1 finale

**Parametri:** usa gli stessi , ,  della cella di configurazione sopra.

> **Nota:** le soglie  nel JSON non sono ancora calibrate sul validation set.
> Per calibrarle: eseguire  con il val-CSV.


In [ ]:
import subprocess, shlex, sys, os, re

REPO = "/content/Visual-Place-Recognition-Project"
env  = {**os.environ, "TQDM_DISABLE": "1"}

SEQUENTIAL_JSON = f"{REPO}/regressors/{MODEL}_sequential_regressors.json"

SEQ_DIR          = f"{DRIVE_ROOT}/adaptive/{RUN_NAME}_sequential"
SEQ_FEATURES_DIR = f"{SEQ_DIR}/features"
SEQ_PREDS_FILT   = f"{SEQ_DIR}/preds_filtered"
SEQ_RERANKED_DIR = f"{SEQ_DIR}/reranked"
for d in [SEQ_FEATURES_DIR, SEQ_PREDS_FILT, SEQ_RERANKED_DIR]:
    os.makedirs(d, exist_ok=True)

def run_step(label, cmd):
    print(f"
{chr(9472)*60}")
    print(f"  {label}")
    print(chr(9472)*60)
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, env=env)
    for line in proc.stdout:
        if re.search(r"^\s*\d+%\|", line) or line.strip() == "":
            continue
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"FALLITO: {label}")
    print("  ✓ completato")

# Step 1: estrazione feature sequenziali (IM top-10 su tutte le query)
run_step("[1/4] Estrazione feature sequenziali (IM top-10)",
    f"python {REPO}/VPR-Adaptive-ReRanking/extract_inliers_sequential.py"
    f" --preds-dir {PREDS_DIR}"
    f" --output-dir {SEQ_FEATURES_DIR}"
    f" --matcher {MATCHER}"
    f" --device cuda")

# Step 2: selezione sequenziale (gate1→gate5→gate10)
run_step("[2/4] Selezione sequenziale",
    f"python {REPO}/VPR-Adaptive-ReRanking/sequential_select.py"
    f" --features-csv {SEQ_FEATURES_DIR}/sequential_features.csv"
    f" --preds-dir {PREDS_DIR}"
    f" --regressors-json {SEQUENTIAL_JSON}"
    f" --matcher {MATCHER}"
    f" --output-dir {SEQ_PREDS_FILT}"
    f" --skipped-file {SEQ_DIR}/skipped.txt")

# Step 3: IM top-20 sulle query selezionate
run_step("[3/4] Reranking (IM top-20)",
    f"python {REPO}/match_queries_preds.py"
    f" --preds-dir {SEQ_PREDS_FILT}"
    f" --out-dir {SEQ_RERANKED_DIR}"
    f" --matcher {MATCHER}"
    f" --device cuda"
    f" --im-size 512"
    f" --num-preds 20")

# Step 4: performance finale
run_step("[4/4] Check performance",
    f"python {REPO}/VPR-Adaptive-ReRanking/check_performance.py"
    f" --preds-dir {PREDS_DIR}"
    f" --reranked-dir {SEQ_RERANKED_DIR}"
    f" --skipped-file {SEQ_DIR}/skipped.txt"
    f" --recall-values 1 5 10 20")

print(f"
{chr(9552)*60}
  SEQUENTIAL PIPELINE COMPLETATA
{chr(9552)*60}")
